In [1]:
import torch
import numpy as np
from torch.utils.data import DataLoader, random_split
from utilities import (
    KolmogorovDataset,
    FNOGenerator,
    FNODiscriminator,
    NavierStokesLoss,
    WGAFNOGPTrainer,
    Rollout,
)
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [2]:
!pip list

Package                   Version
------------------------- ----------------
aiohappyeyeballs          2.6.1
aiohttp                   3.13.3
aiosignal                 1.4.0
annotated-doc             0.0.4
annotated-types           0.7.0
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asciitree                 0.3.3
asttokens                 3.0.1
async-lru                 2.2.0
async-timeout             5.0.1
attrs                     25.4.0
babel                     2.18.0
beautifulsoup4            4.14.3
bleach                    6.3.0
bokeh                     3.9.0
certifi                   2026.2.25
cffi                      2.0.0
cftime                    1.6.5
charset-normalizer        3.4.6
click                     8.3.1
comm                      0.2.3
configmypy                0.2.0
contourpy                 1.3.2
cycler                    0.12.1
debugpy                   1.8.20
decorator     

In [3]:
# ── Device ─────────────────────────────────────────────────
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = "../datasets/Snapshots/snapshots_64x64_use.npy"
H, W      = 64, 64
SEQ_LEN   = 100
EPOCHS    = 2

In [4]:
# ── Hiperparámetros — ajustados para GPU ≤8GB ──────────────
# Con HIDDEN=64, BATCH=16, N_CRITIC=5 el pico estimado es ~5-6GB
# solo para D + GP con create_graph=True.
# Estos valores reducen el pico a ~3-4GB manteniendo capacidad.
BATCH    = 32    # era 16
MODES    = 16    # era 12
HIDDEN   = 64   # era 64
N_LAYERS = 4    # era 4
N_CRITIC = 5    # era 5

In [5]:
# ── Estocasticidad ─────────────────────────────────────────
# z_dim canales de ruido N(0,I) concatenados a wₙ en cada paso.
# z_dim=0 → modo determinista (compatible con código anterior).
# z_dim=4 es buen punto de partida: diversidad sin dominar la señal.
# Si el modelo colapsa a predicciones suaves, reducir ALPHA_MSE a 0.1.
Z_DIM    = 8

In [6]:
# ── Física ─────────────────────────────────────────────────
NU       = .05  # viscosidad cinemática — típica para Kolmogorov turbulento
DT       = 0.1    # paso temporal de los datos

In [7]:
# ── Pérdidas ───────────────────────────────────────────────
LAMBDA_GP = 10.0
ALPHA_MSE = 1.0   # bajar a 0.1 si el generador colapsa a campos suaves
BETA_PHYS = 0.1

In [8]:
# ── Dataset ────────────────────────────────────────────────
dataset   = KolmogorovDataset(DATA_PATH, seq_len=SEQ_LEN)
train_len = int(0.8 * len(dataset))
val_len   = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH, shuffle=False, num_workers=4,
)

2026-03-27 10:41:31,092 | INFO | Dataset: 1152 trayectorias × 90 ventanas = 103,680 muestras | seq_len=10 | H×W=64×64


In [9]:
# ── Modelos ────────────────────────────────────────────────
G = FNOGenerator(
    hidden_ch=HIDDEN, modes1=MODES, modes2=MODES,
    n_layers=N_LAYERS, z_dim=Z_DIM,
)
D = FNODiscriminator(
    seq_len=SEQ_LEN + 1, hidden_ch=HIDDEN,
    modes1=MODES, modes2=MODES, n_layers=N_LAYERS,
)
ns_loss = NavierStokesLoss(H, W, nu=NU, dt=DT, device=DEVICE)

In [10]:
# ── Trainer ────────────────────────────────────────────────
trainer = WGAFNOGPTrainer(
    generator          = G,
    discriminator      = D,
    ns_loss            = ns_loss,
    device             = DEVICE,
    lr_G               = 1e-4,
    lr_D               = 1e-4,
    n_critic           = N_CRITIC,
    lambda_gp          = LAMBDA_GP,
    alpha_mse          = ALPHA_MSE,
    beta_phys          = BETA_PHYS,
    use_scheduler      = True,
    scheduler_patience = 5,
    scheduler_factor   = 0.5,
    log_dir            = "training_logs",   # checkpoints, logs, figuras
    vis_freq           = 1,                # visualizar cada 10 épocas
)

In [11]:
# Opcional: reanudar desde checkpoint
# start_epoch = trainer.load_checkpoint("training_logs/best_checkpoint.pt")

# ── Entrenamiento ──────────────────────────────────────────
history = trainer.fit(train_loader, val_loader, n_epochs=EPOCHS, log_every=10)

Epoch    1/2:   0%|                                    | 0/2592 [00:00<?, ?it/s]/home/joseluis/miniconda3/envs/cfd-project-personal/lib/python3.10/site-packages/torch/autograd/graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
                                                                                

OutOfMemoryError: CUDA out of memory. Tried to allocate 34.00 MiB. GPU 0 has a total capacity of 3.94 GiB of which 22.50 MiB is free. Including non-PyTorch memory, this process has 3.40 GiB memory in use. Of the allocated memory 3.19 GiB is allocated by PyTorch, and 134.42 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
history

In [ ]:
# ── Rollout final ──────────────────────────────────────────
sample = next(iter(val_loader))
w0     = sample[0][:, 0]   # (B, 1, H, W) — condición inicial
w_true = sample[2]         # (B, seq_len+1, H, W) — ground truth

roller = Rollout(G, DEVICE)

# Rollout estocástico — z fresco en cada paso
traj= roller.run(w0, n_steps=100, w_true=w_true)
print("\nRollout estocástico:")
print("  Energía:       ", roller.metrics["energy"])
print("  RMSE:          ", roller.metrics["rmse"])
print("  Error relativo:", roller.metrics["rel_error"])

# Metricas
w_gen = traj_stochastic[:, -1]   # último frame generado (B, H, W)
w_gt  = w_true[:, -1].float()    # último frame ground truth
k_gen, E_gen = WGAFNOGPTrainer.energy_spectrum(w_gen)
k_gt,  E_gt  = WGAFNOGPTrainer.energy_spectrum(w_gt)
print(f"\nEspectro E(k) — k=[1..5]: gen={E_gen[:5].round(4)}, gt={E_gt[:5].round(4)}")
logger.info("Rollout completado. Métricas:")
logger.info(f"Energía: {roller.metrics['energy']}")
logger.info(f"RMSE: {roller.metrics['rmse']}")
logger.info(f"Error relativo: {roller.metrics['rel_error']}")

In [ ]:
seq = traj[0].cpu().numpy()  # (T, H, W)

fig, ax = plt.subplots(figsize=(4, 4))
vmin, vmax = -2, 2
im = ax.imshow(seq[0], cmap='RdBu_r', vmin=vmin, vmax=vmax)
ax.axis('off')

def animate(t):
    im.set_data(seq[t])
    ax.set_title(f't={t}')
    return [im]

ani = animation.FuncAnimation(fig, animate, frames=seq.shape[0], interval=100, blit=True)
ani.save('rollout.gif', writer='pillow')  # Necesita pillow instalado
plt.close(fig)